# 02 — Baseline Model

This notebook trains a fraud detection classifier on `creditcard.csv` and evaluates it as the baseline.  
The saved model is later applied to drift batches in `04_impact_analysis.ipynb`.

**Contents**
1. Setup & data loading
2. Preprocessing
3. Train/test split
4. Handle class imbalance (SMOTE)
5. Train model (Random Forest)
6. Evaluate baseline
7. Save model

## 1. Setup & data loading

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, RocCurveDisplay, PrecisionRecallDisplay
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Paths
os.chdir(r'C:\Users\baxjo\OneDrive\Documenten\GitHub\datadrift-challenge')
os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

RANDOM_STATE = 42

In [ ]:
train_df = pd.read_csv('data/creditcard.csv')
print(f'Loaded: {train_df.shape}')
train_df.head(3)

## 2. Preprocessing

In [ ]:
# Scale Amount and Time — V features are already PCA-transformed
scaler = StandardScaler()

train_df['Amount_scaled'] = scaler.fit_transform(train_df[['Amount']])
train_df['Time_scaled']   = scaler.fit_transform(train_df[['Time']])

# Save the Amount scaler for use on drift batches later
amount_scaler = StandardScaler()
amount_scaler.fit(train_df[['Amount']])
joblib.dump(amount_scaler, 'models/amount_scaler.pkl')

# Define feature set
v_features = [f'V{i}' for i in range(1, 29)]
FEATURES = v_features + ['Amount_scaled', 'Time_scaled']
TARGET   = 'Class'

X = train_df[FEATURES]
y = train_df[TARGET]

print(f'Features: {len(FEATURES)}')
print(f'Class balance: {y.value_counts().to_dict()}')

## 3. Train/test split

In [ ]:
# Stratified split to preserve fraud ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size : {X_train.shape[0]:,} ({y_train.sum()} fraud)')
print(f'Test size  : {X_test.shape[0]:,} ({y_test.sum()} fraud)')
print(f'Train fraud rate: {y_train.mean():.4%}')
print(f'Test fraud rate : {y_test.mean():.4%}')

## 4. Handle class imbalance (SMOTE)

In [ ]:
# SMOTE oversamples the minority class synthetically
# Applied ONLY to training data — never to test data
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'After SMOTE : {pd.Series(y_train_resampled).value_counts().to_dict()}')
print(f'New train size: {X_train_resampled.shape[0]:,}')

## 5. Train model

In [ ]:
# Random Forest — good baseline for imbalanced tabular data
# class_weight='balanced' adds an extra safety net on top of SMOTE
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('Training model...')
model.fit(X_train_resampled, y_train_resampled)
print('Done.')

## 6. Evaluate baseline

In [ ]:
# Predictions
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Core metrics
auc_roc   = roc_auc_score(y_test, y_pred_proba)
f1        = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

print('=== Baseline Model Performance ===')
print(f'AUC-ROC   : {auc_roc:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Save baseline metrics for comparison in notebook 04
baseline_metrics = {
    'batch': 'baseline (test set)',
    'auc_roc': round(auc_roc, 4),
    'f1': round(f1, 4),
    'precision': round(precision, 4),
    'recall': round(recall, 4)
}
pd.DataFrame([baseline_metrics]).to_csv('models/baseline_metrics.csv', index=False)
print('Baseline metrics saved to models/baseline_metrics.csv')

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
ax.set_title('Confusion matrix — baseline model (test set)')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('figures/baseline_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC curve + Precision-Recall curve side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], color='steelblue')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title(f'ROC Curve — AUC = {auc_roc:.4f}')

PrecisionRecallDisplay.from_predictions(y_test, y_pred_proba, ax=axes[1], color='tomato')
axes[1].set_title('Precision-Recall Curve')

plt.suptitle('Baseline model — test set evaluation', fontsize=13)
plt.tight_layout()
plt.savefig('figures/baseline_roc_pr.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['tomato' if i >= len(importances) - 10 else 'steelblue' for i in range(len(importances))]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature importance — baseline Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/baseline_feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 10 most important features:')
print(importances.tail(10).sort_values(ascending=False))

## 7. Save model

In [ ]:
# Save model and feature list — needed for drift batch evaluation in notebook 04
joblib.dump(model, 'models/baseline_model.pkl')
joblib.dump(FEATURES, 'models/feature_list.pkl')

print('Saved:')
print('  models/baseline_model.pkl')
print('  models/feature_list.pkl')
print('  models/amount_scaler.pkl')
print('  models/baseline_metrics.csv')
print(f'\nModel trained on {X_train_resampled.shape[0]:,} samples (after SMOTE)')
print(f'Baseline AUC-ROC: {auc_roc:.4f} | F1: {f1:.4f}')